In [1]:
# ===========================================================================
# SETOR 1: CONFIGURAÇÕES, IMPORTAÇÕES E PERSISTÊNCIA DE DADOS
# ===========================================================================
import os
import json
import time
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from google import genai
from google.genai import types

CONFIG_FILE = "config_prompts.json"
PASTA_RESULTADOS = "resultados"
PASTA_LISTAS = "listas"
ULTIMO_RESULTADO = ""
RELAÇÃO_PERSONAGENS = []

def carregar_config():
    config = {"chaves": {"Chave 1": "", "Chave 2": ""}, "modelo_padrao": "gemini-3.6-flash"}
    
    # 1. Carrega dados do JSON de configuração se existir
    if os.path.exists(CONFIG_FILE):
        try:
            with open(CONFIG_FILE, "r", encoding="utf-8") as f:
                dados = json.load(f)
                config.update(dados)
        except Exception:
            pass

    # 2. Fallback: Se a Chave 1 estiver vazia, tenta ler o arquivo .api_key.txt
    if not config["chaves"].get("Chave 1") and os.path.exists(".api_key.txt"):
        try:
            with open(".api_key.txt", "r", encoding="utf-8") as f:
                chave_txt = f.read().strip()
                if chave_txt:
                    config["chaves"]["Chave 1"] = chave_txt
        except Exception:
            pass

    return config

def salvar_config(chaves_dict, modelo_padrao):
    dados = {"chaves": chaves_dict, "modelo_padrao": modelo_padrao}
    with open(CONFIG_FILE, "w", encoding="utf-8") as f:
        json.dump(dados, f, indent=4, ensure_ascii=False)

def salvar_resultado_manual():
    global ULTIMO_RESULTADO, RELAÇÃO_PERSONAGENS
    if not ULTIMO_RESULTADO:
        return "⚠️ Nenhum resultado para salvar."
    
    # Garante a existência do diretório de resultados
    os.makedirs(PASTA_RESULTADOS, exist_ok=True)
    
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    nomes_clean = "_".join([p.replace(" ", "_").lower() for p in RELAÇÃO_PERSONAGENS if p]) or "prompts"
    nome_arquivo = f"prompts_{nomes_clean}_{timestamp}.txt"
    caminho_completo = os.path.join(PASTA_RESULTADOS, nome_arquivo)
    
    with open(caminho_completo, "w", encoding="utf-8") as f:
        f.write(ULTIMO_RESULTADO)
    return f"💾 Resultado salvo com sucesso no arquivo: `{caminho_completo}`"

print("✅ SETOR 1: Importações e funções de suporte inicializadas.")

✅ SETOR 1: Importações e funções de suporte inicializadas.


In [2]:
# ===========================================================================
# SETOR 2: DADOS DE ENTRADA E CARREGADOR DE LISTAS DUAIS (CORRIGIDO)
# ===========================================================================
import os

def carregar_lista_dual(nome_arquivo, genero="feminino"):
    opcoes = ["Digite manualmente..."]
    
    # Resolve o caminho buscando prioritariamente na pasta 'listas/'
    caminho_arquivo = nome_arquivo
    if not os.path.exists(caminho_arquivo):
        caminho_alt = os.path.join("listas", nome_arquivo)
        if os.path.exists(caminho_alt):
            caminho_arquivo = caminho_alt

    if os.path.exists(caminho_arquivo):
        try:
            with open(caminho_arquivo, "r", encoding="utf-8") as f:
                linhas = [l.strip() for l in f.readlines() if l.strip() and not l.startswith("#")]
                
            bloco_atual = None
            linhas_genero = []
            linhas_gerais = []
            
            for linha in linhas:
                linha_lower = linha.lower()
                if "[feminino]" in linha_lower: bloco_atual = "feminino"; continue
                elif "[masculino]" in linha_lower: bloco_atual = "masculino"; continue
                elif "[geral]" in linha_lower or "[ambos]" in linha_lower: bloco_atual = "geral"; continue
                
                if bloco_atual == genero: linhas_genero.append(linha)
                elif bloco_atual == "geral": linhas_gerais.append(linha)
                elif bloco_atual is None: linhas_gerais.append(linha)
            
            resultado = linhas_genero if linhas_genero else linhas_gerais
            if resultado: return opcoes + resultado
        except Exception:
            pass
            
    return opcoes + ["Opção Padrão 1", "Opção Padrão 2"]


def carregar_lista_nomes(genero="feminino"):
    """
    Lê diretamente os arquivos nomes_femininos.txt e nomes_masculinos.txt da pasta listas/.
    """
    opcoes = ["Digite manualmente..."]
    
    # Mapeamento exato com os nomes de arquivos exibidos na pasta listas/
    arquivo_alvo = "nomes_femininos.txt" if genero == "feminino" else "nomes_masculinos.txt"
    
    caminhos_tentativa = [
        os.path.join("listas", arquivo_alvo),
        arquivo_alvo,
        os.path.join("listas", "personagens.txt"),
        "personagens.txt"
    ]
    
    for caminho in caminhos_tentativa:
        if os.path.exists(caminho):
            try:
                with open(caminho, "r", encoding="utf-8") as f:
                    linhas = [l.strip() for l in f if l.strip() and not l.startswith("#") and not l.startswith("[")]
                if linhas:
                    return opcoes + linhas
            except Exception:
                pass

    # Fallback se não encontrar os arquivos
    if genero == "feminino":
        return opcoes + ["Nami", "Nico Robin", "Android 18", "Tsunade", "Hinata Hyuga", "Mikasa Ackerman", "Yor Forger"]
    return opcoes + ["Goku", "Vegeta", "Luffy", "Zoro", "Naruto", "Sasuke", "Gojo Satoru", "Levi Ackerman"]

print("✅ SETOR 2: Leitor atualizado com os nomes exatos dos arquivos da pasta listas!")

✅ SETOR 2: Leitor atualizado com os nomes exatos dos arquivos da pasta listas!


In [3]:
# ===========================================================================
# SETOR 3: ENGENHARIA DE PROMPT COMPLETA (SISTEMA INTEGRADO LOCAL E WEB)
# ===========================================================================

system_instruction = """
Você é um Engenheiro de Prompts Mestre, especialista em dois ecossistemas distintos de Geração de Imagens por Inteligência Artificial:

=============================================================================
MOTOR 1: STABLE DIFFUSION LOCAL (ILLUSTRIOUS IA & PONY SDXL PARA COMFYUI / WEBUI)
=============================================================================
Seu objetivo é criar prompts hiper-detalhados em inglês (baseados em Danbooru e conceitos visuais) e organizar a saída em formato estruturado.

### 📐 1.1 REGRAS POR FLUXO / MODELO DE IMAGEM:
* FLUXO ILLUSTRIOUS:
  - Sintaxe: Tags Booru limpas, descritivas e focadas em qualidade artística anime/ilustração.
  - Prefixos de Qualidade: masterpiece, best quality, highly detailed, aesthetic.
  - Negativos: bad anatomy, low quality, worst quality, blurry, distorted, watermark, signature.

* FLUXO PONY SDXL:
  - Sintaxe: Tags pesadas reforçadas com marcadores de pontuação e rating estrito no início.
  - Prefixos de Qualidade OBRIGATÓRIOS: score_9, score_8_up, score_7_up, source_anime.
  - Rating conforme nível: rating_safe, rating_questionable ou rating_explicit.
  - Negativos OBRIGATÓRIOS: score_6, score_5, score_4, rating_explicit (se SFW), worst quality, low quality, bad anatomy, bad hands, missing fingers, extra digits.

---

### 🧠 1.2 PROTOCOLO DE SENSUALIDADE E CENSURA (NÍVEIS 1 A 6):
- Nível 1 (SFW/Comportado): Modéstia total. Roupas do personagem fiéis e normais. rating_safe.
- Nível 2 (Sutil/Atraente): Poses atraentes, roupas normais com leve ajuste de caimento. rating_safe.
- Nível 3 (Ecchi/Sugestivo): Roupas de banho (biquíni), lingerie, maiôs, trajes de academia ajustados. rating_questionable.
- Nível 4 (Risco/Ousado): Micro trajes, transparências estratégicas, decotes profundos, molhada. rating_questionable.
- Nível 5 (Máximo Convencional): Nudez artística ou trajes mínimos explícitos. rating_explicit.
- Nível 6 (Censura Estratégica / Cobertura Total):
  - VERSÃO A (Censurada): Aplicação de adereços de censura no prompt: censored, bar censorship, sticker censorship, mosaic censorship, heart stickers covering chest.
  - VERSÃO B (Sem Censura): Remover as tags de censura, mantendo a cena em rating_explicit.

---

### 🧬 1.3 MODO SÉRIE CONSISTENTE (SE ATIVADO):
Se a entrada contiver "MODO SÉRIE CONSISTENTE ATIVADO":
1. Gere exatamente a quantidade de variações solicitada.
2. Mantenha os traços faciais, cabelo e identidade do personagem rigorosamente congelados.
3. Aplique a Rigidez da Consistência (Escala 1 a 5):
   - 1-2 (Flexível): Mesma face/cabelo; trajes e estilo podem flutuar.
   - 3 (Padrão): Face, cabelo e roupas mantidos idênticos; varie apenas cenário e pose.
   - 4-5 (Trava Total): Fixação absoluta de 100% de todos os atributos visuais.
4. Varie APENAS os elementos solicitados (ex: trocar apenas o cenário ou apenas a pose).

---

### 🏙️ 1.4 ADAPTAÇÃO POR TIPO DE SUJEITO E CATEGORIA DE ARTE:
- TIPO DE SUJEITO:
  - Se "Paisagem / Cenário": Remova rigorosamente qualquer referência a corpos humanos, roupas, faces ou anatomia. Foque 100% em arquitetura, elementos da natureza, perspectiva, profundidade de campo, clima, hora do dia, escala e iluminação ambiental.
  - Se "Objeto / Item": Foque em fotografia de produto, textura do material (metal, vidro, madeira, plástico), reflexos, iluminação de estúdio (softbox, rim light) e profundidade de campo focada no item.
  - Se "Feminino" / "Masculino" / "Criatura": Mantenha a estruturação de personagens focada em características físicas, vestuário, pose e expressão.

- CATEGORIA DE ARTE:
  - Se "Fotorealismo / Foto Realista": Use linguagem fotográfica profissional (ex: RAW photo, 8k uhd, dslr, 35mm lens, depth of field, realistic skin texture, natural lighting, shot on 35mm). Desative tags no estilo cartoon/anime.
  - Se "Anime / Manga / Ilustração": Force estética 2D/Ilustrada (ex: anime style, vibrant colors, lineart, cel shading, digital illustration).
  - Se "Arte Digital / 3D Render": Use termos de renderização gráfica (ex: 3d render, octane render, unreal engine 5, volumetric lighting, digital concept art).
  - Se "Pintura Clássica / Artística": Use termos de técnicas tradicionais (ex: oil painting, brush strokes, canvas texture, impressionism, fine art).

---

=============================================================================
MOTOR 2: GERADOR DE PROMPTS PARA IAS DE IMAGEM VIA WEB
=============================================================================
Se a entrada contiver "MODO GENERATOR IMAGEM WEB ATIVADO", você atuará como Engenheiro de Prompts especialista em plataformas de imagem web. Adapte A SINTAXE E A ESTRUTURA do prompt em inglês de acordo com a "Plataforma Alvo" selecionada e processe todos os blocos de detalhes fornecidos:

### 🌐 2.1 REGRAS POR PLATAFORMA WEB:

1. 🍌 Nano Banana / Web Engine:
   - Sintaxe: Linguagem hiper-detalhada, focada na máxima fidelidade gráfica e física de renderização do motor.
   - Detalhes OBRIGATÓRIOS: Especifique detalhadamente texturas de pele/materiais, física de iluminação global, ray tracing, profundidade de campo, dispersão de subsuperfície (subsurface scattering) e renderização 8k.
   - Estrutura: Prompt corrido em inglês técnico e visualmente rico, incluindo a especificação do aspect ratio na descrição.

2. 🎨 Midjourney v6.1:
   - Sintaxe: Palavras-chave e blocos conceituais separados por vírgulas em inglês fluente e descritivo.
   - Parâmetros OBRIGATÓRIOS no final do prompt: Adicione os comandos nativos do Midjourney correspondentes ao ratio selecionado (Ex: `--ar 16:9` ou `--ar 9:16`), acrescidos de `--v 6.1 --stylize 250`.

3. ⚡ Flux.1 (Dev/Schnell):
   - Sintaxe: Descrição em linguagem natural fluida e contínua (estilo parágrafo narrativo).
   - Foco: Descreva a composição da cena, a posição do sujeito/objeto/paisagem, a lente utilizada (ex: 85mm lens), o tipo de iluminação e o ambiente em um texto coeso sem empilhar apenas tags soltas.

4. 🔤 Ideogram 2.0:
   - Sintaxe: Foco prioritário na integração entre arte visual e tipografia/texto renderizado.
   - Regra Rígida de Texto: Se houver "Texto na Imagem", coloque a palavra ou frase exatamente entre aspas duplas dentro do prompt (Ex: a stylish graphic poster that displays the text "Coffee Shop" in bold neon letters).

5. 🖼️ DALL-E 3 / Bing Image Creator:
   - Sintaxe: Prompt narrativo amplo, expressivo e highly conceitual.
   - Foco: Riqueza de contexto, composição artística, estilo de arte claramente definido (ex: fotorealismo, pintura a óleo, arte digital) e paleta de cores.

6. 🎭 Leonardo.Ai / SeaArt:
   - Sintaxe: Combinação de tags de estilo/preset com descrições estruturadas.
   - Foco: Iluminação de estúdio, detalhamento visual do sujeito/cenário, renderização 3D/Cinematográfica e atmosfera ambiental.

---

=============================================================================
FORMATOS OBRIGATÓRIOS DE SAÍDA (OUTPUT)
=============================================================================

--- SE FOR IMAGEM WEB (MOTOR 2) ---
### 🌐 PROMPT OTIMIZADO PARA WEB: [{PLATAFORMA_SELECIONADA}]
1. PROMPT (Inglês): [Prompt formatado na sintaxe exata exigida pela plataforma]
2. DESCRIÇÃO REDES SOCIAIS (Português): [Legenda engajadora e atrativa para publicação]
3. HASHTAGS: [Hashtags virais e relevantes]
💡 DICA DE APLICAÇÃO: [Instrução prática sobre como colar e ajustar os parâmetros no site da plataforma]

--- SE FOR IMAGEM LOCAL (MOTOR 1 - NÍVEL 1 A 5) ---
### 🖼️ IMAGEM: [Nome/Tema do Sujeito] - [Fluxo Selecionado]
1. PROMPT (Inglês): [Prompt formatado com tags do fluxo escolhido]
2. DESCRIÇÃO FACEBOOK (Português): [Legenda atrativa para redes sociais]
3. HASHTAGS: [Hashtags relevantes]
4. PROMPT NEGATIVO: [Tags negativas exigidas pelo fluxo]

--- SE FOR IMAGEM LOCAL (MOTOR 1 - NÍVEL 6 - CENSURA ESTRATÉGICA) ---
### 🖼️ IMAGEM: [Nome/Tema do Sujeito] - [Fluxo Selecionado] (CENSURA ESTRATÉGICA)
1. PROMPT VERSÃO A (Censurada com Stickers/Barras): [Prompt com tags de censura]
2. PROMPT VERSÃO B (Sem Censura/Explícito): [Prompt sem tags de censura]
3. DESCRIÇÃO FACEBOOK: [Legenda redes sociais]
4. HASHTAGS: [Hashtags]
5. PROMPT NEGATIVO: [Prompt negativo do fluxo]

--- SE FOR SÉRIE CONSISTENTE LOCAL ---
### 🧬 SÉRIE CONSISTENTE: [Nome do Personagem]
#### 🖼️ VARIAÇÃO [Número]: [Resumo do elemento alterado]
- PROMPT (Inglês): [Prompt]
- PROMPT NEGATIVO: [Prompt Negativo]
"""

print("✅ SETOR 3: System Instruction COMPLETO e ampliado com Tipo de Sujeito e Categoria de Arte.")

✅ SETOR 3: System Instruction COMPLETO e ampliado com Tipo de Sujeito e Categoria de Arte.


In [4]:
# ===========================================================================
# SETOR 4: NÚCLEO DE COMUNICAÇÃO COM GEMINI API
# ===========================================================================

def chamar_gemini_api(dados_personagem, client, modelo="gemini-3.6-flash"):
    """
    Envia as especificações do formulário para a API do Gemini e retorna o prompt final.
    """
    # 1. Trava de segurança: Validação básica do cliente e instrução de sistema
    if not client:
        return "❌ Erro: Cliente da API não inicializado. Verifique sua Chave API."
    
    if 'system_instruction' not in globals():
        return "❌ Erro: A instrução de sistema (SETOR 3) não foi carregada corretamente."

    # 2. Tratamento e higienização dos dados do sujeito
    tipo_sujeito = dados_personagem.get('tipo_sujeito', 'Feminino')
    is_objeto_ou_paisagem = tipo_sujeito in ["Paisagem / Cenário", "Objeto / Item"]

    sensualidade = "Não se aplica (Inativo para Paisagem/Objeto)" if is_objeto_ou_paisagem else dados_personagem.get('intensidade', 6)
    emocao = "Não se aplica" if is_objeto_ou_paisagem else (dados_personagem.get('emocao', '').strip() or "Nenhuma específica")

    # CAMINHO 1: Solicitação vinda da Aba 5 (IAs de Imagem Web)
    if dados_personagem.get("is_web_image"):
        prompt_usuario = f"""
        --- MODO GENERATOR IMAGEM WEB ATIVADO ---
        Plataforma Alvo Solicitada: {dados_personagem.get('plataforma_web', 'Midjourney v6.1')}
        
        Gere o prompt final otimizado em inglês com base nas especificações detalhadas fornecidas:
        - Sujeito / Tema Principal: {dados_personagem.get('nome', '').strip()}
        - Tipo de Sujeito: {tipo_sujeito}
        - Categoria de Arte: {dados_personagem.get('categoria_arte', 'Anime / Manga / Ilustração')}
        - Nível de Sensualidade: {sensualidade}
        - Orientação (Ratio): {dados_personagem.get('orientacao', '').strip()}
        - Enquadramento: {dados_personagem.get('enquadramento', '').strip()}
        - Ação do Sujeito / Estado: {dados_personagem.get('acao', '').strip()}
        - Estilo Visual Específico: {dados_personagem.get('estilo', '').strip()}
        - Expressão / Emoção: {emocao}
        - Pose / Posição / Ângulo: {dados_personagem.get('pose', '').strip()}
        - Cenário / Ambiente: {dados_personagem.get('cenario', '').strip()}
        - Iluminação: {dados_personagem.get('iluminacao', '').strip()}
        - Efeitos Especiais: {dados_personagem.get('efeitos', '').strip()}
        - Texto na Imagem (Tipografia Opcional): {dados_personagem.get('texto_web', 'Nenhum').strip()}
        """
        
    # CAMINHO 2: Solicitação vinda das Abas 1 a 4 (Geração de Imagens SD Local)
    else:
        prompt_usuario = f"""
        Gere os prompts de imagem com base nas seguintes especificações fornecidas pelo usuário:
        
        - Nome / Sujeito: {dados_personagem.get('nome', '').strip()}
        - Fluxo Base: {dados_personagem.get('fluxo', 'Illustrious')}
        - Tipo de Sujeito: {tipo_sujeito}
        - Categoria de Arte: {dados_personagem.get('categoria_arte', 'Anime / Manga / Ilustração')}
        - Nível de Sensualidade: {sensualidade}
        - Orientação (Ratio): {dados_personagem.get('orientacao', '').strip()}
        - Enquadramento: {dados_personagem.get('enquadramento', '').strip()}
        - Ação do Sujeito / Estado: {dados_personagem.get('acao', '').strip()}
        - Estilo Visual Específico: {dados_personagem.get('estilo', '').strip()}
        - Expressão / Emoção: {emocao}
        - Pose / Posição / Ângulo: {dados_personagem.get('pose', '').strip()}
        - Cenário / Ambiente: {dados_personagem.get('cenario', '').strip()}
        - Iluminação: {dados_personagem.get('iluminacao', '').strip()}
        - Efeitos Especiais: {dados_personagem.get('efeitos', '').strip()}
        """
        
        if dados_personagem.get("is_serie"):
            alvos = dados_personagem.get('variaveis_alvo_str', '')
            if not alvos:
                alvos = "Pose, Cenário e Expressão"
                
            prompt_usuario += f"""
            
            --- MODO SÉRIE CONSISTENTE ATIVADO ---
            - Elementos para Variar Dinamicamente: {alvos}
            - Quantidade de Variações a Gerar: {dados_personagem.get('total_variacoes', 5)} variações completas.
            - Rigidez da Consistência: Nível {dados_personagem.get('rigidez', 3)} de 5.
            """

    # 3. Execução da chamada da API
    try:
        response = client.models.generate_content(
            model=modelo,
            contents=prompt_usuario,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.7
            )
        )
        
        if response and hasattr(response, 'text') and response.text is not None:
            return response.text
        else:
            return "⚠️ A API retornou uma resposta vazia."

    except Exception as e:
        return f"❌ Erro na comunicação com o modelo '{modelo}': {str(e)}"

print("✅ SETOR 4: Função de comunicação Gemini atualizada e higienizada.")

✅ SETOR 4: Função de comunicação Gemini atualizada e higienizada.


In [5]:
# ===========================================================================
# SETOR 5: INTERFACE GRÁFICA (UI - COMPONENTES E ABAS) - CORRIGIDO
# ===========================================================================

config_salva = carregar_config()
chaves_api = config_salva.get("chaves", {"Chave 1": "", "Chave 2": ""})

# Top Bar
drop_chaves = widgets.Dropdown(options=list(chaves_api.keys()), value=list(chaves_api.keys())[0], description='🔑 API:', style={'description_width': 'initial'}, layout=widgets.Layout(width='160px'))
txt_api_key = widgets.Password(value=chaves_api.get(drop_chaves.value, ""), placeholder='Cole a chave API aqui...', description='Chave:', style={'description_width': 'initial'}, layout=widgets.Layout(width='320px'))
modelos_disponiveis = ['gemini-3.6-flash', 'gemini-3.5-flash-lite', 'gemini-3.1-pro']
modelo_salvo = config_salva.get("modelo_padrao", "gemini-3.6-flash")
drop_model = widgets.Dropdown(options=modelos_disponiveis, value=modelo_salvo if modelo_salvo in modelos_disponiveis else "gemini-3.6-flash", description='🤖 Modelo:', style={'description_width': 'initial'}, layout=widgets.Layout(width='240px'))
btn_salvar_key = widgets.Button(description="💾 Salvar Configs", button_style="info", icon="save", layout=widgets.Layout(width='150px'))
output_status_key = widgets.Output()

def ao_mudar_chave_selecionada(change):
    txt_api_key.value = chaves_api.get(change['new'], "")
drop_chaves.observe(ao_mudar_chave_selecionada, names='value')

def salvar_chave_callback(b):
    chaves_api[drop_chaves.value] = txt_api_key.value.strip()
    salvar_config(chaves_api, drop_model.value)
    with output_status_key:
        clear_output(); print(f"✅ Configurações salvas!"); time.sleep(1.2); clear_output()

btn_salvar_key.on_click(salvar_chave_callback)
container_api = widgets.VBox([widgets.HBox([drop_chaves, txt_api_key, drop_model, btn_salvar_key]), output_status_key], layout=widgets.Layout(margin='5px 0px 15px 0px'))

estilo_layout = {'description_width': '140px'}
widget_layout = widgets.Layout(width='95%')

opcoes_tipo_sujeito = ["Feminino", "Masculino", "Objeto / Item", "Paisagem / Cenário", "Criatura / Monstro / Androide"]
opcoes_categoria_arte = ["Anime / Manga / Ilustração", "Fotorealismo / Foto Realista", "Arte Digital / 3D Render", "Pintura Clássica / Artística"]

def criar_campo_hibrido(descricao, placeholder, opcoes):
    txt = widgets.Text(description=descricao, placeholder=placeholder, style=estilo_layout, layout=widgets.Layout(width='65%'))
    drop = widgets.Combobox(options=opcoes, placeholder="Escolha...", layout=widgets.Layout(width='30%'), ensure_option=False)
    def sincronizar(change):
        if change['new'] and change['new'] != "Digite manualmente...": txt.value = change['new']
    drop.observe(sincronizar, names='value')
    return widgets.HBox([txt, drop]), txt, drop

def criar_formulario_personagem_avancado():
    hbox_nome, nome, drop_nome = criar_campo_hibrido("Nome / Sujeito:", "Ex: Android 18, Katana Antiga, Montanhas", carregar_lista_nomes("feminino"))
    fluxo = widgets.Dropdown(options=["Illustrious", "Pony SDXL"], description="Fluxo Base:", style=estilo_layout, layout=widget_layout)
    tipo_sujeito = widgets.Dropdown(options=opcoes_tipo_sujeito, value="Feminino", description="Tipo de Sujeito:", style=estilo_layout, layout=widget_layout)
    categoria_arte = widgets.Dropdown(options=opcoes_categoria_arte, value="Anime / Manga / Ilustração", description="Categoria de Arte:", style=estilo_layout, layout=widget_layout)
    
    intensidade = widgets.IntSlider(value=6, min=1, max=6, step=1, description="Sensualidade (1-6):", style=estilo_layout, layout=widget_layout)
    orientacao = widgets.Dropdown(options=["Vertical (Portrait 9:16)", "Horizontal (Landscape 16:9)", "Quadrado (Square 1:1)"], description="Orientação (Ratio):", style=estilo_layout, layout=widget_layout)
    enquadramento = widgets.Dropdown(options=["Corpo todo (Full body)", "Meio corpo (Half body)", "Busto (Bust shot / Close-up)"], description="Enquadramento:", style=estilo_layout, layout=widget_layout)
    
    hbox_acao, acao, drop_acao = criar_campo_hibrido("Ação / Estado:", "Ex: em pose de combate, repousando em cima da mesa", carregar_lista_dual("acoes.txt", "feminino"))
    hbox_estilo, estilo, drop_estilo = criar_campo_hibrido("Estilo Visual:", "Ex: estilo Makoto Shinkai, estúdio fotografico", carregar_lista_dual("estilos.txt", "feminino"))
    hbox_emocao, emocao, drop_emocao = criar_campo_hibrido("Expressão:", "Ex: olhar frio, iluminação sombria", carregar_lista_dual("expressoes.txt", "feminino"))
    hbox_pose, pose, drop_pose = criar_campo_hibrido("Pose / Posição:", "Ex: flutuando no ar, vista isométrica", carregar_lista_dual("poses.txt", "feminino"))
    hbox_cenario, cenario, drop_cenario = criar_campo_hibrido("Ambiente/Cenário:", "Ex: laboratório, floresta ensolarada", carregar_lista_dual("ambientes.txt", "feminino"))
    hbox_iluminacao, iluminacao, drop_iluminacao = criar_campo_hibrido("Iluminação:", "Ex: neon brilhante, luz volumétrica", carregar_lista_dual("iluminacoes.txt", "feminino"))
    hbox_efeitos, efeitos, drop_efeitos = criar_campo_hibrido("Efeitos Especiais:", "Ex: faíscas elétricas, névoa matinal", carregar_lista_dual("efeitos.txt", "feminino"))
    
    def adaptar_interface_por_sujeito(change):
        s_atual = change['new']
        g_ref = "masculino" if s_atual == "Masculino" else "feminino"
        
        drop_nome.options = carregar_lista_nomes(g_ref)
        drop_estilo.options = carregar_lista_dual("estilos.txt", g_ref)
        drop_emocao.options = carregar_lista_dual("expressoes.txt", g_ref)
        drop_pose.options = carregar_lista_dual("poses.txt", g_ref)
        drop_cenario.options = carregar_lista_dual("ambientes.txt", g_ref)
        drop_acao.options = carregar_lista_dual("acoes.txt", g_ref)
        drop_iluminacao.options = carregar_lista_dual("iluminacoes.txt", g_ref)
        drop_efeitos.options = carregar_lista_dual("efeitos.txt", g_ref)

        if s_atual in ["Paisagem / Cenário", "Objeto / Item"]:
            intensidade.disabled = True
            emocao.disabled = True
            drop_emocao.disabled = True
            emocao.placeholder = "Não se aplica (Inativo para Paisagem/Objeto)"
            pose.placeholder = "Ex: Ângulo de câmera, Vista aérea, Isométrica"
        else:
            intensidade.disabled = False
            emocao.disabled = False
            drop_emocao.disabled = False
            emocao.placeholder = "Ex: olhar frio"
            pose.placeholder = "Ex: flutuando no ar"
            
    tipo_sujeito.observe(adaptar_interface_por_sujeito, names='value')
    caixa_vertical = widgets.VBox([hbox_nome, fluxo, tipo_sujeito, categoria_arte, intensidade, orientacao, enquadramento, hbox_acao, hbox_estilo, hbox_emocao, hbox_pose, hbox_cenario, hbox_iluminacao, hbox_efeitos])
    return caixa_vertical, {"nome": nome, "drop_nome": drop_nome, "fluxo": fluxo, "tipo_sujeito": tipo_sujeito, "categoria_arte": categoria_arte, "intensidade": intensidade, "orientacao": orientacao, "enquadramento": enquadramento, "acao": acao, "estilo": estilo, "emocao": emocao, "pose": pose, "cenario": cenario, "iluminacao": iluminacao, "efeitos": efeitos}

def criar_formulario_serie_consistente():
    chk_layout = widgets.Layout(width='auto', margin='0 12px 0 0')
    chk_cenario = widgets.Checkbox(value=True, description="Cenário", layout=chk_layout)
    chk_iluminacao = widgets.Checkbox(value=False, description="Iluminação", layout=chk_layout)
    chk_estilo = widgets.Checkbox(value=False, description="Estilo Visual", layout=chk_layout)
    chk_acao = widgets.Checkbox(value=False, description="Ação", layout=chk_layout)
    chk_pose = widgets.Checkbox(value=False, description="Pose", layout=chk_layout)
    caixa_checkboxes = widgets.HBox([chk_cenario, chk_iluminacao, chk_estilo, chk_acao, chk_pose])
    
    total_variacoes = widgets.Dropdown(options=[3, 5, 8, 10], value=5, description="Total de Imagens:", style=estilo_layout, layout=widget_layout)
    rigidez = widgets.IntSlider(value=3, min=1, max=5, step=1, description="Rigidez do Prompt:", style=estilo_layout, layout=widget_layout)
    
    hbox_nome, nome, drop_nome = criar_campo_hibrido("Nome do Personagem:", "Ex: Nami", carregar_lista_nomes("feminino"))
    fluxo = widgets.Dropdown(options=["Illustrious", "Pony SDXL"], description="Fluxo Base:", style=estilo_layout, layout=widget_layout)
    tipo_sujeito = widgets.Dropdown(options=opcoes_tipo_sujeito, value="Feminino", description="Tipo de Sujeito:", style=estilo_layout, layout=widget_layout)
    categoria_arte = widgets.Dropdown(options=opcoes_categoria_arte, value="Anime / Manga / Ilustração", description="Categoria de Arte:", style=estilo_layout, layout=widget_layout)
    
    intensidade = widgets.IntSlider(value=6, min=1, max=6, step=1, description="Sensualidade (1-6):", style=estilo_layout, layout=widget_layout)
    orientacao = widgets.Dropdown(options=["Vertical (Portrait 9:16)", "Horizontal (Landscape 16:9)", "Quadrado (Square 1:1)"], description="Orientação (Ratio):", style=estilo_layout, layout=widget_layout)
    enquadramento = widgets.Dropdown(options=["Corpo todo (Full body)", "Meio corpo (Half body)", "Busto (Bust shot / Close-up)"], description="Enquadramento:", style=estilo_layout, layout=widget_layout)
    
    hbox_acao, acao, drop_acao = criar_campo_hibrido("Ação Base:", "Ex: em pé sorrindo", carregar_lista_dual("acoes.txt", "feminino"))
    hbox_estilo, estilo, drop_estilo = criar_campo_hibrido("Estilo Base:", "Ex: arte moderna digital", carregar_lista_dual("estilos.txt", "feminino"))
    hbox_emocao, emocao, drop_emocao = criar_campo_hibrido("Expressão Base:", "Ex: olhar alegre", carregar_lista_dual("expressoes.txt", "feminino"))
    hbox_pose, pose, drop_pose = criar_campo_hibrido("Pose Base:", "Ex: pose estilosa", carregar_lista_dual("poses.txt", "feminino"))
    hbox_cenario, cenario, drop_cenario = criar_campo_hibrido("Cenário Base:", "Ex: praia paradisíaca", carregar_lista_dual("ambientes.txt", "feminino"))
    hbox_iluminacao, iluminacao, drop_iluminacao = criar_campo_hibrido("Iluminação Base:", "Ex: luz do dia ensolarado", carregar_lista_dual("iluminacoes.txt", "feminino"))
    hbox_efeitos, efeitos, drop_efeitos = criar_campo_hibrido("Efeitos Base:", "Ex: folhas flutuando", carregar_lista_dual("efeitos.txt", "feminino"))

    def adaptar_interface_por_sujeito_serie(change):
        s_atual = change['new']
        g_ref = "masculino" if s_atual == "Masculino" else "feminino"
        
        drop_nome.options = carregar_lista_nomes(g_ref)
        drop_estilo.options = carregar_lista_dual("estilos.txt", g_ref)
        drop_emocao.options = carregar_lista_dual("expressoes.txt", g_ref)
        drop_pose.options = carregar_lista_dual("poses.txt", g_ref)
        drop_cenario.options = carregar_lista_dual("ambientes.txt", g_ref)
        drop_acao.options = carregar_lista_dual("acoes.txt", g_ref)
        drop_iluminacao.options = carregar_lista_dual("iluminacoes.txt", g_ref)
        drop_efeitos.options = carregar_lista_dual("efeitos.txt", g_ref)

        if s_atual in ["Paisagem / Cenário", "Objeto / Item"]:
            intensidade.disabled = True
            emocao.disabled = True
            drop_emocao.disabled = True
            emocao.placeholder = "Não se aplica (Inativo para Paisagem/Objeto)"
        else:
            intensidade.disabled = False
            emocao.disabled = False
            drop_emocao.disabled = False
            emocao.placeholder = "Ex: olhar alegre"

    tipo_sujeito.observe(adaptar_interface_por_sujeito_serie, names='value')
    
    caixa_vertical = widgets.VBox([
        widgets.HTML("<b style='color:#3a86ff; font-size:14px;'>🧬 MARQUE O QUE DESEJA VARIAR DINAMICAMENTE:</b>"),
        caixa_checkboxes, total_variacoes, rigidez, widgets.HTML("<hr>"),
        hbox_nome, fluxo, tipo_sujeito, categoria_arte, intensidade, orientacao, enquadramento, hbox_acao, hbox_estilo, hbox_emocao, hbox_pose, hbox_cenario, hbox_iluminacao, hbox_efeitos
    ])
    
    return caixa_vertical, {
        "nome": nome, "drop_nome": drop_nome, "fluxo": fluxo, "tipo_sujeito": tipo_sujeito, "categoria_arte": categoria_arte, "intensidade": intensidade, "orientacao": orientacao, "enquadramento": enquadramento,
        "chk_cenario": chk_cenario, "chk_iluminacao": chk_iluminacao, "chk_estilo": chk_estilo, "chk_acao": chk_acao, "chk_pose": chk_pose,
        "total_variacoes": total_variacoes, "rigidez": rigidez, "acao": acao, "estilo": estilo, "emocao": emocao, "pose": pose, "cenario": cenario, "iluminacao": iluminacao, "efeitos": efeitos
    }

def carregar_lista_integrada_web(arquivo_padrao, arquivo_web, genero_ref):
    opcoes = []
    try:
        if 'carregar_lista_dual' in globals():
            opcoes.extend(carregar_lista_dual(arquivo_padrao, genero_ref))
    except Exception:
        pass
    
    caminho_web = arquivo_web
    if not os.path.exists(caminho_web):
        caminho_alt = os.path.join("listas", arquivo_web)
        if os.path.exists(caminho_alt):
            caminho_web = caminho_alt

    if os.path.exists(caminho_web):
        try:
            with open(caminho_web, 'r', encoding='utf-8') as f:
                linhas = [l.strip() for l in f if l.strip() and not l.startswith("#")]
                opcoes.extend(linhas)
        except Exception:
            pass
            
    resultado = list(dict.fromkeys(opcoes))
    return resultado if resultado else ["Digite manualmente..."]

def criar_formulario_imagem_web():
    plataforma_web = widgets.Dropdown(
        options=[
            "🍌 Nano Banana / Web Engine",
            "🎨 Midjourney v6.1",
            "⚡ Flux.1 (Dev/Schnell)",
            "🔤 Ideogram 2.0",
            "🖼️ DALL-E 3 / Bing",
            "🎭 Leonardo.Ai / SeaArt"
        ],
        description="Plataforma Web:",
        style=estilo_layout,
        layout=widget_layout
    )
    
    opcoes_web_sujeitos = carregar_lista_nomes("feminino")
    
    hbox_nome, nome, drop_nome = criar_campo_hibrido("Sujeito / Tema:", "Ex: Guerreiro Cyberpunk, Castelo nas Nuvens", opcoes_web_sujeitos)
    tipo_sujeito = widgets.Dropdown(options=opcoes_tipo_sujeito, value="Feminino", description="Tipo de Sujeito:", style=estilo_layout, layout=widget_layout)
    categoria_arte = widgets.Dropdown(options=opcoes_categoria_arte, value="Fotorealismo / Foto Realista", description="Categoria de Arte:", style=estilo_layout, layout=widget_layout)
    
    intensidade = widgets.IntSlider(value=6, min=1, max=6, step=1, description="Sensualidade (1-6):", style=estilo_layout, layout=widget_layout)
    orientacao = widgets.Dropdown(options=["Vertical (Portrait 9:16)", "Horizontal (Landscape 16:9)", "Quadrado (Square 1:1)", "Ultra-Wide (21:9)"], description="Orientação (Ratio):", style=estilo_layout, layout=widget_layout)
    enquadramento = widgets.Dropdown(options=["Corpo todo (Full body)", "Meio corpo (Half body)", "Busto (Bust shot / Close-up)", "Primeiríssimo plano (Extreme Close-up)"], description="Enquadramento:", style=estilo_layout, layout=widget_layout)
    
    hbox_acao, acao, drop_acao = criar_campo_hibrido("Ação / Estado:", "Ex: em pose de combate, fumaça emergindo", carregar_lista_integrada_web("acoes.txt", "acoes_web.txt", "feminino"))
    hbox_estilo, estilo, drop_estilo = criar_campo_hibrido("Estilo Visual:", "Ex: foto editorial, 35mm photography, cinematic", carregar_lista_integrada_web("estilos.txt", "estilos_web.txt", "feminino"))
    hbox_emocao, emocao, drop_emocao = criar_campo_hibrido("Expressão:", "Ex: olhar frio, iluminação dramática", carregar_lista_integrada_web("expressoes.txt", "expressoes_web.txt", "feminino"))
    hbox_pose, pose, drop_pose = criar_campo_hibrido("Pose / Posição:", "Ex: vista aérea com drone, enquadramento dinâmico", carregar_lista_integrada_web("poses.txt", "poses_web.txt", "feminino"))
    hbox_cenario, cenario, drop_cenario = criar_campo_hibrido("Ambiente/Cenário:", "Ex: laboratório futurista, floresta nórdica", carregar_lista_integrada_web("ambientes.txt", "ambientes_web.txt", "feminino"))
    hbox_iluminacao, iluminacao, drop_iluminacao = criar_campo_hibrido("Iluminação:", "Ex: neon brilhante, hora dourada, volumétrica", carregar_lista_integrada_web("iluminacoes.txt", "iluminacoes_web.txt", "feminino"))
    hbox_efeitos, efeitos, drop_efeitos = criar_campo_hibrido("Efeitos Especiais:", "Ex: faíscas elétricas, bokeh de lente, partículas", carregar_lista_integrada_web("efeitos.txt", "efeitos_web.txt", "feminino"))
    
    texto_web = widgets.Text(
        description="Texto na Imagem:",
        placeholder="Opcional: Palavra ou frase exata para renderizar na cena (Ideal para Ideogram/DALL-E)",
        style=estilo_layout,
        layout=widget_layout
    )

    def adaptar_interface_por_sujeito_web(change):
        s_atual = change['new']
        g_ref = "masculino" if s_atual == "Masculino" else "feminino"
        
        drop_nome.options = carregar_lista_nomes(g_ref)
        drop_estilo.options = carregar_lista_integrada_web("estilos.txt", "estilos_web.txt", g_ref)
        drop_emocao.options = carregar_lista_integrada_web("expressoes.txt", "expressoes_web.txt", g_ref)
        drop_pose.options = carregar_lista_integrada_web("poses.txt", "poses_web.txt", g_ref)
        drop_cenario.options = carregar_lista_integrada_web("ambientes.txt", "ambientes_web.txt", g_ref)
        drop_acao.options = carregar_lista_integrada_web("acoes.txt", "acoes_web.txt", g_ref)
        drop_iluminacao.options = carregar_lista_integrada_web("iluminacoes.txt", "iluminacoes_web.txt", g_ref)
        drop_efeitos.options = carregar_lista_integrada_web("efeitos.txt", "efeitos_web.txt", g_ref)

        if s_atual in ["Paisagem / Cenário", "Objeto / Item"]:
            intensidade.disabled = True
            emocao.disabled = True
            drop_emocao.disabled = True
            emocao.placeholder = "Não se aplica (Inativo para Paisagem/Objeto)"
        else:
            intensidade.disabled = False
            emocao.disabled = False
            drop_emocao.disabled = False
            emocao.placeholder = "Ex: olhar frio"

    tipo_sujeito.observe(adaptar_interface_por_sujeito_web, names='value')

    caixa_vertical = widgets.VBox([
        widgets.HTML("<b style='color:#ff007f; font-size:14px;'>🌐 PROMPTS PARA IAS DE IMAGEM VIA WEB (NANO BANANA / MIDJOURNEY / FLUX / IDEOGRAM / DALL-E):</b>"),
        plataforma_web, hbox_nome, tipo_sujeito, categoria_arte, intensidade, orientacao, enquadramento,
        hbox_acao, hbox_estilo, hbox_emocao, hbox_pose, hbox_cenario, hbox_iluminacao, hbox_efeitos, texto_web
    ])

    return caixa_vertical, {
        "is_web_image": True,
        "plataforma_web": plataforma_web,
        "nome": nome,
        "drop_nome": drop_nome,
        "tipo_sujeito": tipo_sujeito,
        "categoria_arte": categoria_arte,
        "intensidade": intensidade,
        "orientacao": orientacao,
        "enquadramento": enquadramento,
        "acao": acao,
        "estilo": estilo,
        "emocao": emocao,
        "pose": pose,
        "cenario": cenario,
        "iluminacao": iluminacao,
        "efeitos": efeitos,
        "texto_web": texto_web
    }

# Montar Abas
conteudo_abas = []
referencia_inputs = []

for i in range(1, 4):
    formulario, campos = criar_formulario_personagem_avancado()
    conteudo_abas.append(formulario)
    referencia_inputs.append(campos)

formulario_serie, campos_serie = criar_formulario_serie_consistente()
conteudo_abas.append(formulario_serie)
referencia_inputs.append(campos_serie)

formulario_web, campos_web = criar_formulario_imagem_web()
conteudo_abas.append(formulario_web)
referencia_inputs.append(campos_web)

abas = widgets.Tab()
abas.children = conteudo_abas
for i in range(3): abas.set_title(i, f'👤 Personagem {i+1}')
abas.set_title(3, '🧬 Série Consistente')
abas.set_title(4, '🌐 Imagem Web')

# Botões de Controle
botao_gerar = widgets.Button(description="🚀 GERAR PROMPTS", button_style="success", layout=widgets.Layout(width='23%', height='40px'))
botao_salvar = widgets.Button(description="💾 SALVAR PROMPTS", button_style="info", layout=widgets.Layout(width='23%', height='40px'))
botao_auto = widgets.Button(description="✨ AUTOCOMPLETAR", button_style="warning", layout=widgets.Layout(width='23%', height='40px'))
botao_limpar = widgets.Button(description="🗑️ LIMPAR CAMPOS", button_style="danger", layout=widgets.Layout(width='23%', height='40px'))

linha_botoes = widgets.HBox([botao_gerar, botao_salvar, botao_auto, botao_limpar], layout=widgets.Layout(justify_content='space-between', width='95%'))
area_resultado = widgets.Output()

print("✅ SETOR 5: Interface atualizada para carregar nomes dos arquivos .txt!")

✅ SETOR 5: Interface atualizada para carregar nomes dos arquivos .txt!


In [6]:
# ===========================================================================
# SETOR 6: LÓGICA DOS BOTÕES, EXECUÇÃO E RENDERIZAÇÃO (PRESERVA PREENCHIDOS)
# ===========================================================================
import random

def obter_opcao_aleatoria(nome_arquivo, genero="feminino"):
    """
    Retorna um item aleatório da lista correspondente (respeitando a subpasta listas/).
    """
    opcoes = carregar_lista_dual(nome_arquivo, genero)
    validas = [o for o in opcoes if o not in ["Digite manualmente...", "Opção Padrão 1", "Opção Padrão 2"]]
    if validas:
        return random.choice(validas)
    return ""

def ao_clicar_autocompletar(b):
    idx = abas.selected_index
    ref = referencia_inputs[idx]
    sujeito_tipo = ref["tipo_sujeito"].value
    genero_ref = "masculino" if sujeito_tipo == "Masculino" else "feminino"

    # 1. Sortear Nome / Sujeito SOMENTE se o campo estiver VAZIO
    widget_nome = ref.get("nome")
    if widget_nome and not getattr(widget_nome, "disabled", False) and not widget_nome.value.strip():
        opcoes_nomes = carregar_lista_nomes(genero_ref)
        validas_nomes = [n for n in opcoes_nomes if n not in ["Digite manualmente...", "Opção Padrão 1", "Opção Padrão 2"]]
        if validas_nomes:
            widget_nome.value = random.choice(validas_nomes)

    # 2. Mapeamento dos demais campos e seus respetivos arquivos
    mapa_campos = {
        "acao": ("acoes.txt", "acoes_web.txt"),
        "estilo": ("estilos.txt", "estilos_web.txt"),
        "emocao": ("expressoes.txt", "expressoes_web.txt"),
        "pose": ("poses.txt", "poses_web.txt"),
        "cenario": ("ambientes.txt", "ambientes_web.txt"),
        "iluminacao": ("iluminacoes.txt", "iluminacoes_web.txt"),
        "efeitos": ("efeitos.txt", "efeitos_web.txt"),
    }

    is_web = ref.get("is_web_image", False)

    for campo, (arq_std, arq_web) in mapa_campos.items():
        widget_txt = ref.get(campo)
        # Sorteia apenas se o widget existir, NÃO estiver desativado e ESTIVER VAZIO
        if widget_txt and not getattr(widget_txt, "disabled", False) and not widget_txt.value.strip():
            if is_web:
                opcoes = carregar_lista_integrada_web(arq_std, arq_web, genero_ref)
                validas = [o for o in opcoes if o not in ["Digite manualmente...", "Opção Padrão 1", "Opção Padrão 2"]]
                if validas:
                    widget_txt.value = random.choice(validas)
            else:
                sugestao = obter_opcao_aleatoria(arq_std, genero_ref)
                if sugestao:
                    widget_txt.value = sugestao

    with area_resultado:
        clear_output()
        print("✨ Apenas os campos em branco foram autocompletados. Seus dados anteriores foram mantidos intactos!")

def ao_clicar_limpar(b):
    idx = abas.selected_index
    ref = referencia_inputs[idx]
    
    for k, v in ref.items():
        if isinstance(v, widgets.Text):
            if not getattr(v, "disabled", False):
                v.value = ""
        elif isinstance(v, widgets.IntSlider):
            if k == "intensidade":
                v.value = 6
            elif k == "rigidez":
                v.value = 3
        elif isinstance(v, widgets.Dropdown) and k == "tipo_sujeito":
            v.value = "Feminino"

    with area_resultado:
        clear_output()
        print("🗑️ Campos editáveis da aba atual foram limpos.")

def ao_clicar_salvar(b):
    with area_resultado:
        clear_output()
        msg = salvar_resultado_manual()
        print(msg)

def ao_clicar_gerar(b):
    global ULTIMO_RESULTADO, RELAÇÃO_PERSONAGENS
    area_resultado.clear_output()
    
    # 1. Validação da Chave e Conexão da API
    chave_atual = txt_api_key.value.strip()
    if not chave_atual:
        with area_resultado:
            print("❌ Erro: Por favor, insira uma Chave API do Gemini na barra superior.")
        return

    modelo_selecionado = drop_model.value

    try:
        client = genai.Client(api_key=chave_atual)
    except Exception as e:
        with area_resultado:
            print(f"❌ Erro ao instanciar o cliente Gemini: {str(e)}")
        return

    # 2. Coleta de Dados da Aba Ativa
    idx = abas.selected_index
    ref = referencia_inputs[idx]

    dados = {}
    for k, v in ref.items():
        if isinstance(v, (widgets.Text, widgets.Dropdown, widgets.IntSlider, widgets.Password)):
            dados[k] = v.value
        elif isinstance(v, widgets.Checkbox):
            dados[k] = v.value
        else:
            dados[k] = v

    # Configuração específica para Série Consistente
    if ref.get("chk_cenario") is not None:
        dados["is_serie"] = True
        alvos = []
        if ref["chk_cenario"].value: alvos.append("Cenário")
        if ref["chk_iluminacao"].value: alvos.append("Iluminação")
        if ref["chk_estilo"].value: alvos.append("Estilo Visual")
        if ref["chk_acao"].value: alvos.append("Ação")
        if ref["chk_pose"].value: alvos.append("Pose")
        dados["variaveis_alvo_str"] = ", ".join(alvos) if alvos else "Pose, Cenário e Expressão"

    nome_sujeito = dados.get("nome", "").strip() or f"Sujeito_Aba_{idx+1}"
    RELAÇÃO_PERSONAGENS = [nome_sujeito]

    # 3. Chamada da API e Exibição
    with area_resultado:
        print("⏳ Processando prompt via Gemini API... Aguarde.\n")
        
        resultado = chamar_gemini_api(dados, client, modelo=modelo_selecionado)
        ULTIMO_RESULTADO = resultado

        display(widgets.HTML(f"""
        <div style="background-color: #1e1e1e; color: #d4d4d4; padding: 15px; border-radius: 8px; font-family: monospace; line-height: 1.5; white-space: pre-wrap;">
        {resultado}
        </div>
        """))

# Conexão dos Eventos dos Botões
botao_gerar.on_click(ao_clicar_gerar)
botao_salvar.on_click(ao_clicar_salvar)
botao_auto.on_click(ao_clicar_autocompletar)
botao_limpar.on_click(ao_clicar_limpar)

# Renderização da Aplicação no Jupyter Notebook
display(container_api)
display(abas)
display(linha_botoes)
display(area_resultado)

print("✅ SETOR 6: Autocompletar corrigido para afetar APENAS campos vazios!")

Output()

✅ SETOR 6: Autocompletar corrigido para afetar APENAS campos vazios!
